In [ ]:
import numpy as np

In [ ]:
class GridWorld:
    def __init__(self, rows=4, cols=4, start=(0, 0), goal=(3, 3), trap=(2, 2)):
        self.rows = rows
        self.cols = cols
        self.start = start
        self.goal = goal
        self.trap = trap
        self.n_states = rows * cols
        self.n_actions = 4
        self.state = None
        self.reset()

    def reset(self):
        self.state = self._pos_to_state(self.start)
        return self.state

    def step(self, action):
        row, col = self._state_to_pos(self.state)

        if action == 0:
            row = max(0, row - 1)
        elif action == 1:
            row = min(self.rows - 1, row + 1)
        elif action == 2:
            col = max(0, col - 1)
        elif action == 3:
            col = min(self.cols - 1, col + 1)

        next_state = self._pos_to_state((row, col))
        self.state = next_state

        if (row, col) == self.goal:
            return next_state, 1.0, True
        if (row, col) == self.trap:
            return next_state, -1.0, True

        return next_state, -0.04, False

    def _pos_to_state(self, pos):
        return pos[0] * self.cols + pos[1]

    def _state_to_pos(self, state):
        return divmod(state, self.cols)

In [ ]:
class QLearningAgent:
    def __init__(self, n_states, n_actions, learning_rate=0.1, discount_factor=0.95, epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.01):
        self.n_states = n_states
        self.n_actions = n_actions
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.q_table = np.zeros((n_states, n_actions))
        self.rewards_history = []

    def choose_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return np.argmax(self.q_table[state])

    def update(self, state, action, reward, next_state, done):
        best_next_action = np.argmax(self.q_table[next_state])
        target = reward if done else reward + self.discount_factor * self.q_table[next_state, best_next_action]
        self.q_table[state, action] += self.learning_rate * (target - self.q_table[state, action])

    def train(self, env, episodes=500):
        self.rewards_history = []

        for _ in range(episodes):
            state = env.reset()
            done = False
            total_reward = 0

            while not done:
                action = self.choose_action(state)
                next_state, reward, done = env.step(action)
                self.update(state, action, reward, next_state, done)
                state = next_state
                total_reward += reward

            self.rewards_history.append(total_reward)
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def predict(self, state):
        return np.argmax(self.q_table[state])

    def get_policy(self):
        return np.argmax(self.q_table, axis=1)

In [ ]:
env = GridWorld(rows=4, cols=4, start=(0, 0), goal=(3, 3), trap=(2, 2))

agent = QLearningAgent(
    n_states=env.n_states,
    n_actions=env.n_actions,
    learning_rate=0.1,
    discount_factor=0.95,
    epsilon=1.0,
    epsilon_decay=0.995,
    epsilon_min=0.01
)

In [ ]:
agent.train(env, episodes=1000)

print("Q-Table:")
print(agent.q_table)

In [ ]:
print("Learned Policy:")
print(agent.get_policy().reshape(env.rows, env.cols))

In [ ]:
action_symbols = {0: "U", 1: "D", 2: "L", 3: "R"}

policy = agent.get_policy()
policy_grid = np.array([action_symbols[a] for a in policy], dtype=object).reshape(env.rows, env.cols)

goal_r, goal_c = env.goal
trap_r, trap_c = env.trap

policy_grid[goal_r, goal_c] = "G"
policy_grid[trap_r, trap_c] = "T"

print(policy_grid)

In [ ]:
state = env.reset()
done = False
path = [env._state_to_pos(state)]

while not done:
    action = agent.predict(state)
    next_state, reward, done = env.step(action)
    path.append(env._state_to_pos(next_state))
    state = next_state

print("Path:", path)
print("Final Reward:", reward)